# ** Gemini**


# Importar librerías

In [3]:
!pip install google-cloud-storage


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:

import pandas as pd

import re
import os
from sklearn.utils import shuffle
import json
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import numpy as np
import ast



In [4]:

from google.cloud import storage

from google import genai
from google.genai import types
from google.genai.types import (
    GenerateContentConfig,
    ThinkingConfig,
    SafetySetting,
    CreateBatchJobConfig,
    JobState,
    HttpOptions,
    EmbedContentConfig,
)

# Funciones

Se proporciona la siguiente función que calcula de forma automatica el coste asociado a la peticion a Gemini. Se recomienda usarla al final de cada sección.

In [5]:
def calcular_coste(tokens_input, tokens_output, batch=False):
    # Precios por 1M tokens para el modelo "Gemini 2.5 Flash"
    precio_input = 0.30  # Precio para entrada de texto
    # precio_input_batch = 0.15   # Precio con 50% de descuento para batch
    
    precio_output = 2.50  # Precio para salida de texto
    # precio_output_batch = 1.25   # Precio con 50% de descuento para batch
    
    # Determinamos el precio de la entrada (input)
    precio_input = precio_input/2 if batch else precio_input
    
    # Determinamos el precio de la salida (output)
    precio_output = precio_output/2 if batch else precio_output
    
    # Calculamos el coste de los tokens de entrada en millones
    coste_input = (tokens_input / 1_000_000) * precio_input
    
    # Calculamos el coste de los tokens de salida en millones
    coste_output = (tokens_output / 1_000_000) * precio_output
    
    # Sumar los costes de input y output
    coste_total = coste_input + coste_output
    
    return round(coste_total, 4)


# Importando/Leyendo Datasets

In [ ]:
GEMINI_API_KEY=''
MODEL_ID = "gemini-3-flash-preview"

In [7]:
# Cargar el fichero
file_path = 'economia_actual_2026.txt'

# Carga el fichero
with open(file_path, 'r', encoding='utf-8') as f:
    articulo = f.read()

# Resumen

## 1. Realiza con Gemini el resumen del artícuylo

In [12]:
def generar_resumenes(texto):
    # Inicializamos el cliente con la API Key definida previamente
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    # Diseñamos el prompt
    prompt_template = """
    Actúa como un analista económico experto. Tu tarea es leer el siguiente texto y generar un resumen conciso destacando:
    1. El estado macroeconómico global (inflación, tasas, crecimiento regional).
    2. El impacto de la tecnología (digitalización e IA).
    3. Las tendencias en sostenibilidad y energía.
    
    Texto a resumir:
    {texto}
    """
    
    # Formateamos el prompt con el artículo real
    prompt_final = prompt_template.format(texto=texto)
    
    # Llamamos al modelo
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_final,
    )

    return response, prompt_final

# Ejecución
response, prompt = generar_resumenes(articulo)
print(response.text)

Como analista económico, presento el siguiente resumen ejecutivo sobre las tendencias y el estado actual de la economía global, basado en el texto proporcionado:

### 1. Estado Macroeconómico Global
La economía mundial atraviesa una **transformación estructural** caracterizada por la búsqueda de estabilidad tras sucesivas crisis. 
*   **Política Monetaria e Inflación:** Aunque la inflación tiende a moderarse, los bancos centrales mantienen una postura prudente con tasas de interés elevadas, lo que limita el consumo y la inversión privada. 
*   **Crecimiento Regional:** Se observa un desempeño heterogéneo. **Estados Unidos** muestra resiliencia, mientras que **Europa** enfrenta dificultades estructurales. **Asia** se consolida como el motor global, con una India en expansión frente a una desaceleración controlada de China. **América Latina** depende de la exportación de materias primas y se beneficia de la regionalización de cadenas de suministro (*nearshoring*).
*   **Desafíos Fiscales

In [13]:
response,prompt = generar_resumenes(articulo)

## 2. Calcula costest de input y output.


resumen, prompt5 = ...

In [14]:
# Extraemos los tokens de los metadatos de la respuesta de Gemini
tokens_input = response.usage_metadata.prompt_token_count
tokens_output = response.usage_metadata.candidates_token_count

# Calculamos el coste usando tu función
coste = calcular_coste(tokens_input, tokens_output)

print(f"Tokens de entrada: {tokens_input}")
print(f"Tokens de salida: {tokens_output}")
print(f"Coste estimado de la petición: ${coste}")

Tokens de entrada: 1227
Tokens de salida: 655
Coste estimado de la petición: $0.002


In [15]:
# ==========================================
# MÉTODO 1: EVALUACIÓN DINÁMICA (5 PASOS)
# ==========================================
def generar_preguntas_complejas(articulo):
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = f"""
    Genera 3 preguntas abiertas y complejas sobre el artículo y sus respuestas.
    Devuelve estrictamente un JSON con esta estructura: [{{"pregunta": "...", "respuesta_articulo": "..."}}]
    Artículo: {articulo}
    """
    res = client.models.generate_content(
        model=MODEL_ID, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    return json.loads(res.text)

def responder_pregunta(resumen, datos_preguntas):
    client = genai.Client(api_key=GEMINI_API_KEY)
    preguntas = [p["pregunta"] for p in datos_preguntas]
    prompt = f"""
    Responde a estas preguntas basándote SOLO en el resumen.
    Devuelve estrictamente un JSON con esta estructura: [{{"pregunta": "...", "respuesta_resumen": "..."}}]
    Resumen: {resumen}
    Preguntas: {preguntas}
    """
    res = client.models.generate_content(
        model=MODEL_ID, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    return json.loads(res.text)

def generar_veredictos_complejos(datos_preguntas, respuestas_resumen):
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt = f"""
    Compara la respuesta esperada del artículo con la respuesta del resumen.
    Asigna una puntuación del 0 al 10 sobre qué tan bien el resumen captura la información.
    Devuelve estrictamente un JSON con esta estructura: [{{"pregunta": "...", "puntuacion": 8, "justificacion": "..."}}]
    Respuestas Artículo: {datos_preguntas}
    Respuestas Resumen: {respuestas_resumen}
    """
    res = client.models.generate_content(
        model=MODEL_ID, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    return json.loads(res.text)

def generar_puntuacion(articulo, resumen):
    datos_preguntas = generar_preguntas_complejas(articulo)
    respuestas_resumen = responder_pregunta(resumen, datos_preguntas)
    veredictos = generar_veredictos_complejos(datos_preguntas, respuestas_resumen)
    return datos_preguntas, respuestas_resumen, veredictos

def calcular_concision(articulo, resumen):
    palabras_art = len(articulo.split())
    palabras_res = len(resumen.split())
    return round((palabras_res / palabras_art) * 100, 2) if palabras_art > 0 else 0

def calcular_puntuacion_final(articulo, resumen, datos_preguntas, veredictos, concision):
    puntuaciones = [v.get("puntuacion", 0) for v in veredictos]
    media = sum(puntuaciones) / len(puntuaciones) if puntuaciones else 0
    
    print("===== RESULTADO DE LA EVALUACIÓN (5 PASOS) =====")
    print(f"Métrica de Concisión: El resumen es un {concision}% del tamaño original.")
    print(f"Puntuación Global de Calidad: {media:.2f} / 10\n")
    
    print("--- Detalle por Pregunta ---")
    for v in veredictos:
        print(f"Pregunta: {v.get('pregunta')}")
        print(f"Puntuación: {v.get('puntuacion')}/10 | Justificación: {v.get('justificacion')}\n")
        
    return media

# ==========================================
# MÉTODO 2: TU EVALUACIÓN ESTÁTICA
# ==========================================
def evaluar_resumen(resumen, articulo_original):
    client = genai.Client(api_key=GEMINI_API_KEY)
    prompt_evaluacion = f"""
    Eres un auditor de contenido. Revisa el 'Artículo Original' y luego el 'Resumen'.
    Responde a las siguientes 3 preguntas complejas para evaluar la calidad del resumen:
    
    1. ¿Menciona el resumen explícitamente la reconfiguración de las cadenas de suministro (nearshoring) que se destaca en el original?
    2. ¿Capta el resumen el contraste entre el crecimiento de Estados Unidos y los desafíos de Europa?
    3. ¿Refleja el resumen la preocupación por la deuda pública global y la sostenibilidad fiscal?
    
    Artículo Original:
    {articulo_original}
    
    Resumen a evaluar:
    {resumen}
    
    Devuelve tu análisis indicando 'SÍ/NO' para cada pregunta y una brevísima justificación.
    """
    eval_response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_evaluacion,
    )
    return eval_response.text

## 3. Mediante la realización de preguntas complejas, evalua el resumen
Revisa el punto 2.2 del notebook Introduccion a prompting-Gemini 2

Respuesta: 

In [16]:
texto_resumen = response.text
# A. Ejecutar tu evaluación original (Preguntas fijas)
print("\n" + "="*50)
print("EVALUACIÓN ESTÁTICA (TU FUNCIÓN ORIGINAL)")
print("="*50)
resultado_estatico = evaluar_resumen(texto_resumen, articulo)
print(resultado_estatico)

# B. Ejecutar la evaluación de 5 pasos (Preguntas dinámicas)
print("\n" + "="*50)
print("EVALUACIÓN DINÁMICA (DIAGRAMA DE 5 PASOS)")
print("="*50)
datos_preguntas, respuestas_resumen, veredictos = generar_puntuacion(articulo, texto_resumen)
concision = calcular_concision(articulo, texto_resumen)
media_final = calcular_puntuacion_final(articulo, texto_resumen, datos_preguntas, veredictos, concision)


EVALUACIÓN ESTÁTICA (TU FUNCIÓN ORIGINAL)
Aquí tienes mi evaluación como auditor de contenido:

**1. ¿Menciona el resumen explícitamente la reconfiguración de las cadenas de suministro (nearshoring) que se destaca en el original?**
*   **SÍ.** El resumen incluye un punto específico bajo el apartado de "Comercio" donde menciona la regionalización de las cadenas de suministro y utiliza explícitamente el término *nearshoring*.

**2. ¿Capta el resumen el contraste entre el crecimiento de Estados Unidos y los desafíos de Europa?**
*   **SÍ.** En la sección de "Crecimiento Regional", el resumen destaca textualmente la resiliencia de Estados Unidos frente a los retos estructurales que enfrenta Europa, manteniendo la comparativa del original.

**3. ¿Refleja el resumen la preocupación por la deuda pública global y la sostenibilidad fiscal?**
*   **SÍ.** El resumen menciona que la deuda pública global ha alcanzado niveles históricos y que la sostenibilidad fiscal se ha convertido en una priorid